# Notebook 02 — Data Validation, Schema Conformance, and Quality Assurance
**Component 03: Hybrid Explainable Contamination Risk Prediction Framework**

### Research Purpose
Performs rigorous schema auditing, physical domain boundary validation (e.g. 0 ≤ pH ≤ 14), temporal continuity checks, and missing value profiling. Crucially, statistical outliers are **flagged rather than deleted**, because severe contamination events naturally present as physical outliers in sensor streams.

### 🔗 Linked Source Modules & Core Implementations
| Source Module | Core Classes / Functions | Primary Responsibility |
| :--- | :--- | :--- |
| [`src/data_validation.py`](../src/data_validation.py) | `EnvironmentalDataValidator`, `validate_dataframe()` | Audits physical sensor bounds, missingness, sampling continuity, and flags extreme events non-destructively. |
| [`src/data_loader.py`](../src/data_loader.py) | `load_raw_data()` | Ingests raw sensor streams from `data/raw/`. |
| [`src/utils.py`](../src/utils.py) | `get_project_root()`, `load_feature_config()` | Supplies physical boundary threshold limits from configuration. |


In [1]:
# =========================================================================
# 1. ENVIRONMENT BOOTSTRAP & LINKED MODULE IMPORTS
# Linked Modules: src/data_loader.py, src/data_validation.py, src/utils.py
# =========================================================================
import sys
from pathlib import Path
import pandas as pd

cur = Path.cwd().resolve()
root_dir = cur if (cur / 'config' / 'project_config.json').exists() else cur.parent
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from src.data_loader import load_raw_data
from src.data_validation import EnvironmentalDataValidator

# Load raw observations into memory
df = load_raw_data()
print(f'Loaded raw dataset with shape: {df.shape}')


Loaded raw dataset with shape: (140160, 21)


In [2]:
# =========================================================================
# 2. EXECUTE DATA QUALITY & PHYSICAL BOUND AUDIT
# Linked Class: EnvironmentalDataValidator from src/data_validation.py
# =========================================================================
validator = EnvironmentalDataValidator()
report = validator.generate_full_validation_report(df)

print('=== DATA VALIDATION AUDIT REPORT ===')
print(f'Total Records: {report["total_records_analyzed"]:,}')
print(f'Schema Conformance: {report["schema_check"]["passed"]}')
print(f'Missing Value Count: {report["missingness"]["total_missing_cells"]}')
print(f'Extreme Events Flagged (Preserved): {report["extreme_events_flagged"]["total_flagged_rows"]:,} ({report["extreme_events_flagged"]["flagged_ratio_pct"]}%)')
print(f'Overall Audit Status: {report["overall_status"]}')


=== DATA VALIDATION AUDIT REPORT ===
Total Records: 140,160
Schema Conformance: True
Missing Value Count: 0
Extreme Events Flagged (Preserved): 26,991 (19.257%)
Overall Audit Status: PASSED


In [3]:
# =========================================================================
# 3. INSPECT PHYSICAL BOUNDARY CONFORMANCE TABLE
# =========================================================================
pd.DataFrame(report['physical_range_audit']).T


,min_allowed,max_allowed,observed_min,observed_max,violations_count,violations_pct
ph,0.0,14.0,1.00,11.63,0.0,0.0
turbidity_ntu,0.0,1000.0,10.00,850.00,0.0,0.0
tds_ppm,0.0,5000.0,77.50,1271.20,0.0,0.0
ec_us_cm,0.0,10000.0,146.50,2606.00,0.0,0.0
water_temp_c,5.0,50.0,22.11,30.47,0.0,0.0
rainfall_mm,0.0,250.0,0.00,117.81,0.0,0.0


### Validation Summary
All physical constraints (pH 0-14, non-negative turbidity, positive EC and TDS) are strictly verified. Zero unexpected gaps exist in the 15-minute sampling schedule. All extreme events are preserved for supervised training.
